<a href="https://colab.research.google.com/github/Naaao9999/shikoku-economic-analysis/blob/main/02_apl_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02_apl_analysis: 地域間産業連関表を用いたサプライチェーン構造の多角的分離分析

## 1. プロジェクト概要
本ノートブックでは、1985年から2005年までの20年間における日本の地域間産業連関表を対象に、サプライチェーンの物理的な「長さ」を示す指標である平均波及長（APL: Average Propagation Length）を算出・分析します。

特に四国地域を中心とした国内サプライチェーンの変遷を、階層ベイズモデル（Hierarchical Bayesian Model）による要因分解と、空間統計学的（GIS）**な可視化を用いて多角的に解明することを目的としています。

---

## 2. 分析の手順

### ① APL（平均波及長）の計測
投入係数行列 $A$ およびレオンチェフ逆行列 $L = (I - A)^{-1}$ を用い、生産波及が部門間を通過する際の「平均的な工程数」を算出します。
$$APL = \frac{L(L-I)}{L-I}$$
> ※本プログラムでは、分母が 0 となる要素（自己ループのみ、または波及がない経路）を適切に除外・処理するロジックを実装しています。

### ② 階層ベイズモデルによる要因分解
APLの変動を「年次効果」「地域効果」「産業効果」およびそれらの交互作用（Interaction Effects）に分離します。これにより、特定の地域や産業が全国的なトレンドからどのように逸脱（Deviation）しているかを定量的に評価します。

- **定式化**:
  $$\mu = \alpha + \beta_{year} + \gamma_{region} + \delta_{sector} + \zeta_{reg \times year} + \eta_{reg \times sec} + \theta_{year \times sec}$$
- **推論エンジン**: `PyMC` によるマルコフ連鎖モンテカルロ法（MCMC/NUTSアルゴリズム）

### ③ 空間分布アニメーション (GIS)
`GeoPandas` を用い、20年間にわたる波及構造の変化を日本地図上にマッピングします。時系列の変化を直感的に捉えるためのGIFアニメーション生成機能を備えています。

---

## 3. 分析パイプライン
本ノートブックは、前工程 `01_data_preprocessing` で生成された「部門統合済み表」を自動的に読み込んで動作します。

1. **Data Loading**: `data/interim/` より統合済みIO表を取得
2. **APL Calculation**: 後方連関（需要牽引）および前方連関（供給波及）APLの算出
3. **Statistical Analysis**: 階層ベイズモデルによる事後分布の推定と収束診断
4. **Visualization**:
    - 産業別交互作用のキャタピラープロット
    - 第三次産業比率とAPLの相関散布図
    - 地域集約マクロ分析の再現
5. **Geospatial Analysis**: APL空間分布の年次推移アニメーション生成

---

## 4. 技術スタック
- **Data Handling**: `pandas`, `numpy`, `xarray`, `openpyxl`
- **Probabilistic Programming**: `PyMC`, `ArviZ`
- **Geospatial Analysis**: `geopandas`, `imageio`, `mpl_toolkits.axes_grid1`
- **Visualization**: `matplotlib`, `seaborn`, `adjustText`

---

> **Note**:
> 本ノートブックで用いるデータは、経済産業省公表の「地域間産業連関表」に基づき、独自に部門統合（31部門）を施した研究用データセットです。部門統合は01_data_preprocessing.ipynbを使用して実装して下さい。

## 環境セットアップ

In [ ]:
# ===== Graphvizのインストール（モデル図作成用） =====
!apt-get install -y graphviz > /dev/null 2>&1
!pip install graphviz -q
print("✓ Graphviz installed")

In [ ]:
# ===== 環境セットアップ =====
"""
APL分析 - Notebook版
地域間産業連関表を用いた平均経路長（APL）分析
"""

# ライブラリインストール
!pip3 install matplotlib-fontja -q
!pip3 install adjustText -q
!pip uninstall -y arviz xarray -q
!pip install arviz xarray -q

# 2. 標準ライブラリ
import os
import pickle
import logging
import glob
from pathlib import Path
from typing import Tuple, List, Optional, Dict, Any
from dataclasses import dataclass, field

# 3. データ処理・数値計算
import pandas as pd
import numpy as np
import xarray as xr

# 4. 統計・ベイズ推定
from scipy.stats import pearsonr
import pymc as pm
import arviz as az

# 5. 地図・GIS関連
import geopandas as gpd
import imageio
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# 6. 可視化・表示
import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント対応
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, Normalize
from adjustText import adjust_text
from IPython.display import Image as IPImage, display

# # Google Driveマウント：Colabで使用する場合
# from google.colab import drive
# drive.mount('/content/drive')

# ロギング設定
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Setup completed")

In [ ]:
@dataclass
class Config:
    """プロジェクト設定 - 公開用最終版"""

    # 1. パスの相対化（重要）
    base_path: Path = field(default_factory=lambda: Path("."))
    data_dir: str = "output/tables"
    save_dir: str = "output/figs"
    table_dir: str = "output/result_tables"
    geojson_path: str = "data/mapping/prefectures.geojson"


    # 地域定義
    regions: List[str] = field(default_factory=lambda: [
        "北海道", "東北", "関東", "中部", "近畿",
        "中国", "四国", "九州", "沖縄"
    ])

    # 第三次産業リスト
    tertiary_sectors: List[str] = field(default_factory=lambda: [
        "電力", "ガス・熱供給", "水道・廃棄物処理",
        "商業", "金融・保険", "不動産", "運輸",
        "その他の情報通信", "公務", "教育・研究",
        "医療・保健・社会保障・介護",
        "広告・対事業所サービス", "対個人サービス", "その他"
    ])

    # 年次とファイルの対応
    years: Dict[str, str] = field(default_factory=lambda: {})

    # 産業分類（後で設定）
    sectors: List[str] = field(default_factory=list)

    # プロット設定
    figure_dpi: int = 300
    base_font_size: int = 12

    # ベイズモデル設定
    mcmc_draws: int = 1000
    mcmc_tune: int = 2000
    mcmc_chains: int =  2
    target_accept: float = 0.96

    def __post_init__(self):
        """初期化後の処理"""
        # ディレクトリ作成
        os.makedirs(self.save_dir, exist_ok=True)
        os.makedirs(self.table_dir, exist_ok=True)

        # 年次とファイルパス設定
        if not self.years:
            self.years = {
                'S60': os.path.join(self.data_dir, 'S60_integrated.xlsx'),
                'H2': os.path.join(self.data_dir, 'H2_integrated.xlsx'),
                'H7': os.path.join(self.data_dir, 'H7_integrated.xlsx'),
                'H17': os.path.join(self.data_dir, 'H17_integrated.xlsx')
            }

        # matplotlibグローバル設定
        plt.rcParams.update({
            'font.size': self.base_font_size,
            'axes.titlesize': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 12,
            'ytick.labelsize': 12,
            'legend.fontsize': 11,
            'figure.facecolor': 'white',
            'axes.facecolor': 'white'
        })

class YearLabelConverter:
    """年次ラベル変換ユーティリティ"""

    YEAR_MAP = {
        1985: "S60", 1990: "H2", 1995: "H7", 2005: "H17"
    }

    @classmethod
    def to_japanese(cls, year) -> str:
        """西暦を和暦ラベルに変換"""
        if isinstance(year, str):
            return year
        return cls.YEAR_MAP.get(year, str(year))

# 設定インスタンス作成
config = Config()
print("✓ Configuration loaded")
print(f"  - Data directory: {config.data_dir}")
print(f"  - Save directory: {config.save_dir}")
print(f"  - Regions: {len(config.regions)}")

## データ構造設定

In [ ]:
# ===== データ構造クラス =====

@dataclass
class IOTableData:
    """産業連関表データ"""
    transaction_matrix: pd.DataFrame
    gross_output: pd.Series
    labels: List[str]
    year: str

    def __post_init__(self):
        """データ検証"""
        assert len(self.transaction_matrix) == len(self.gross_output), \
            "取引行列と生産額の次元不一致"
        assert len(self.labels) == len(self.transaction_matrix), \
            "ラベル数と行列次元不一致"

@dataclass
class LinkageResults:
    """連関分析結果"""
    backward_linkage: np.ndarray
    forward_linkage: np.ndarray
    apl_matrix: np.ndarray
    input_coefficient: np.ndarray
    leontief_inverse: np.ndarray

    def get_backward_mean(self) -> np.ndarray:
        """後方APLの平均を返す"""
        return np.nanmean(self.apl_matrix, axis=0)

    def get_forward_mean(self) -> np.ndarray:
        """前方APLの平均を返す"""
        return np.nanmean(self.apl_matrix, axis=1)

@dataclass
class AnalysisResults:
    """全体の分析結果"""
    apl_results: Dict[str, Dict[str, List[float]]] = field(default_factory=dict)
    sector_apl_results: Dict[str, Dict[str, List[float]]] = field(default_factory=dict)
    shikoku_sector_apl: Dict[str, Dict[str, List[float]]] = field(default_factory=dict)
    tertiary_ratios: Dict[str, List[float]] = field(default_factory=dict)
    raw_linkages: Dict[str, LinkageResults] = field(default_factory=dict)
    io_data: Dict[str, IOTableData] = field(default_factory=dict)

@dataclass
class BayesianModelResults:
    """ベイズ推定結果を格納するクラス"""
    trace: az.InferenceData
    model: Optional[pm.Model] = None
    summary: Optional[pd.DataFrame] = None
    waic: Optional[float] = None
    loo: Optional[float] = None

print("✓ Data structure classes defined")

## データ読み込み

In [ ]:
# ===== データ読み込みクラス =====

class IOTableLoader:
    """産業連関表の読み込みクラス"""

    def __init__(self, valid_regions: List[str], valid_sectors: List[str]):
        self.valid_regions = valid_regions
        self.valid_sectors = valid_sectors
        self.logger = logging.getLogger(self.__class__.__name__)

    def load(self, filepath: str, year: str) -> IOTableData:
        """
        Excelファイルから産業連関表を読み込む

        Parameters
        ----------
        filepath : str
            Excelファイルのパス
        year : str
            年次ラベル

        Returns
        -------
        IOTableData
            構造化された産業連関表データ
        """
        self.logger.info(f"Loading {year} from {filepath}")

        # Excel読み込み
        df = pd.read_excel(filepath, header=None)

        # 列ラベルの抽出
        col_regions = df.iloc[3, 4:].tolist()
        col_sectors = df.iloc[5, 4:].tolist()
        col_labels = [f"{r}_{s}".strip() for r, s in zip(col_regions, col_sectors)]

        # 行ラベルの抽出
        row_regions = df.iloc[6:, 1].tolist()
        row_sectors = df.iloc[6:, 3].tolist()
        row_labels = [f"{r}_{s}".strip() for r, s in zip(row_regions, row_sectors)]

        # 取引行列の抽出
        data = df.iloc[6:, 4:]
        data.index = row_labels
        data.columns = col_labels
        data = data.apply(pd.to_numeric, errors='coerce').fillna(0)

        # 負の値をゼロに置換
        negative_count = (data < 0).sum().sum()
        if negative_count > 0:
            self.logger.warning(f"Replaced {negative_count} negative values with zero in {year}")
            data[data < 0] = 0

        # 生産額の抽出
        gross_output = self._extract_gross_output(df, col_labels)

        # フィルタリング
        valid_labels = [l for l in row_labels if self._is_valid_label(l)]
        filtered_matrix = data.loc[valid_labels, valid_labels]
        filtered_output = gross_output[valid_labels]

        self.logger.info(f"Filtered to {len(valid_labels)} valid entries")

        return IOTableData(
            transaction_matrix=filtered_matrix,
            gross_output=filtered_output,
            labels=valid_labels,
            year=year
        )

    def _extract_gross_output(self, df: pd.DataFrame, col_labels: List[str]) -> pd.Series:
        """地域内生産額を抽出"""
        gross_output_row_idx = None

        for i, (region, sector) in enumerate(zip(df.iloc[6:, 1], df.iloc[6:, 3])):
            if "地域計" in str(region) and "地域内生産額" in str(sector):
                gross_output_row_idx = i + 6
                break

        if gross_output_row_idx is None:
            raise ValueError("地域計_地域内生産額の行が見つかりません")

        gross_output = df.iloc[gross_output_row_idx, 4:]
        gross_output.index = col_labels
        return gross_output.astype(float)

    def _is_valid_label(self, label: str) -> bool:
        """ラベルが有効かどうか判定"""
        exclude_keywords = [
            "計", "屑", "付加価値", "消費", "余剰",
            "引当", "税", "補助金", "生産額", "需要"
        ]

        for keyword in exclude_keywords:
            if keyword in label:
                return False

        try:
            region, sector = label.split('_', 1)
            return (region in self.valid_regions and
                    sector in self.valid_sectors)
        except ValueError:
            return False

class SectorExtractor:
    """産業分類の抽出ユーティリティ"""

    @staticmethod
    def extract_from_excel(filepath: str,
                          start_sector: str = "農林水産業",
                          end_sector: str = "その他") -> List[str]:
        """Excelファイルから産業分類リストを抽出"""
        df = pd.read_excel(filepath, header=None)
        row_6 = df.iloc[5]

        try:
            start_idx = row_6[row_6 == start_sector].index[0]
            end_idx = row_6[row_6 == end_sector].index[0]
            sectors = row_6.iloc[start_idx:end_idx + 1].tolist()
            logger.info(f"Extracted {len(sectors)} sectors")
            return sectors
        except IndexError:
            raise ValueError(f"産業名 '{start_sector}' または '{end_sector}' が見つかりません")

print("✓ Data loader classes defined")

## APL計算

In [ ]:
# ===== APL計算クラス =====

class APLCalculator:
    """APL計算エンジン"""

    def __init__(self, epsilon: float = 1e-10):
        self.epsilon = epsilon
        self.logger = logging.getLogger(self.__class__.__name__)

    def calculate(self, transaction_matrix: pd.DataFrame,
                  gross_output: pd.Series) -> LinkageResults:
        """
        APLと連関指標を計算

        Parameters
        ----------
        transaction_matrix : pd.DataFrame
            取引行列Z
        gross_output : pd.Series
            地域内生産額ベクトル

        Returns
        -------
        LinkageResults
            計算結果
        """
        # 生産額ベクトルの準備（ゼロ除算回避）
        gross = gross_output.replace(0, self.epsilon).values.astype(float)
        Z = transaction_matrix.values.astype(float)

        # 投入係数行列A = Z / x^T
        A = Z / gross

        # 産出係数行列B = Z^T / x
        B = (Z.T / gross).T

        # 単位行列
        I = np.eye(A.shape[0])

        # レオンチェフ逆行列L = (I - A)^(-1)
        L = self._compute_inverse(I - A, "Leontief inverse L")

        # ゴーシュ逆行列G = (I - B)^(-1)
        G = self._compute_inverse(I - B, "Ghosh inverse G")

        # APL行列の計算
        H = L.dot(L - I)
        denominator = L - I
        denominator[denominator == 0] = np.nan
        apl_matrix = H / denominator

        # 連関指標
        backward_linkage = L.sum(axis=0)  # 列和
        forward_linkage = G.sum(axis=1)   # 行和

        return LinkageResults(
            backward_linkage=backward_linkage,
            forward_linkage=forward_linkage,
            apl_matrix=apl_matrix,
            input_coefficient=A,
            leontief_inverse=L
        )

    def _compute_inverse(self, matrix: np.ndarray, name: str) -> np.ndarray:
        """逆行列を安全に計算"""
        try:
            inv_matrix = np.linalg.solve(matrix, np.eye(matrix.shape[0]))
            self.logger.debug(f"{name} computed using solve()")
            return inv_matrix
        except np.linalg.LinAlgError:
            self.logger.warning(f"{name} is singular, using pseudo-inverse")
            return np.linalg.pinv(matrix)

class RegionalAggregator:
    """地域・産業別の集計クラス"""

    @staticmethod
    def aggregate_by_region(values: np.ndarray, labels: List[str],
                           output: pd.Series, regions: List[str]) -> List[float]:
        """地域別に生産額加重平均で集計"""
        region_values = []

        for region in regions:
            indices = [i for i, lab in enumerate(labels)
                      if lab.startswith(region + '_')]

            if not indices:
                region_values.append(np.nan)
                continue

            vals = [values[i] for i in indices]
            weights = [output.iloc[i] for i in indices]
            total_weight = sum(weights)

            if total_weight > 0:
                weighted_avg = sum(v * w for v, w in zip(vals, weights)) / total_weight
                region_values.append(weighted_avg)
            else:
                region_values.append(np.nan)

        return region_values

    @staticmethod
    def aggregate_by_sector(values: np.ndarray, labels: List[str],
                           output: pd.Series, sectors: List[str]) -> List[float]:
        """産業別に生産額加重平均で集計"""
        sector_values = []

        for sector in sectors:
            indices = [i for i, lab in enumerate(labels)
                      if lab.endswith('_' + sector)]

            if not indices:
                sector_values.append(np.nan)
                continue

            vals = [values[i] for i in indices]
            weights = [output.iloc[i] for i in indices]
            total_weight = sum(weights)

            if total_weight > 0:
                weighted_avg = sum(v * w for v, w in zip(vals, weights)) / total_weight
                sector_values.append(weighted_avg)
            else:
                sector_values.append(np.nan)

        return sector_values

    @staticmethod
    def aggregate_matrix_by_region(Z: np.ndarray, x: np.ndarray, labels: List[str], regions: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        """
        取引行列Zと生産額xを地域単位（9x9）に集約する（論文図2再現用）
        """
        n_reg = len(regions)
        Z_agg = np.zeros((n_reg, n_reg))
        x_agg = np.zeros(n_reg)

        # 地域ごとのインデックスを抽出
        region_map = {}
        for i, label in enumerate(labels):
            r = label.split('_')[0]
            if r not in region_map: region_map[r] = []
            region_map[r].append(i)

        for i, r_from in enumerate(regions):
            idx_from = region_map.get(r_from, [])
            if not idx_from: continue
            x_agg[i] = x.iloc[idx_from].sum()
            for j, r_to in enumerate(regions):
                idx_to = region_map.get(r_to, [])
                if not idx_to: continue
                # ブロック行列の合計をとる
                Z_agg[i, j] = Z[np.ix_(idx_from, idx_to)].sum()

        return Z_agg, x_agg

print("✓ APL calculator classes defined")

## ベイズ分析クラス

In [ ]:
# ===== ベイズ分析クラス一式 =====

class RegionalAPLAnalyzer:
    """地域・産業別APL分析のためのベイズ推定クラス"""

    def __init__(self, model_builder: 'BayesianModelBuilder', config: 'Config'):
        self.model_builder = model_builder
        self.config = config

    def prepare_data(self, analysis_results: 'AnalysisResults', linkage_type: str) -> Dict:
        """
        論文のパネル構造 (年次×地域×産業) にデータを成形
        """
        years = sorted(list(analysis_results.io_data.keys()))
        regions = self.config.regions
        sectors = self.config.sectors
        key = 'back' if linkage_type == 'backward' else 'forw'

        y_list, r_idx, t_idx, s_idx = [], [], [], []

        for t_val, year in enumerate(years):
            # 産業・地域別の生データ（NM次元）からAPLを抽出
            io_data = analysis_results.io_data[year]
            linkage = analysis_results.raw_linkages[year]

            # 各部門の平均波及長を算出 [cite: 105, 162]
            apl_matrix = linkage.apl_matrix
            if linkage_type == 'backward':
                # 後方APL: 列方向の平均 [cite: 66, 77]
                apl_values = np.nanmean(apl_matrix, axis=0)
            else:
                # 前方APL: 行方向の平均 [cite: 91, 102]
                apl_values = np.nanmean(apl_matrix, axis=1)

            for i, label in enumerate(io_data.labels):
                try:
                    reg, sec = label.split('_', 1)
                    if reg in regions and sec in sectors:
                        val = apl_values[i]
                        if not np.isnan(val):
                            y_list.append(val)
                            r_idx.append(regions.index(reg))
                            t_idx.append(t_val)
                            s_idx.append(sectors.index(sec))
                except ValueError:
                    continue

        return {
            'y': np.array(y_list),
            'region_idx': np.array(r_idx),
            'time_idx': np.array(t_idx),
            'sector_idx': np.array(s_idx),
            'n_regions': len(regions),
            'n_times': len(years),
            'n_sectors': len(sectors)
        }

    def analyze(self, analysis_results: 'AnalysisResults', linkage_type: str = 'backward') -> 'BayesianModelResults':
        """
        APLデータのベイズ分析を実行
        """
        # データ準備 (引数をanalysis_resultsオブジェクトに変更)
        data = self.prepare_data(analysis_results, linkage_type)

        # モデル構築 (論文式14の全交互作用を含む)
        model = self.model_builder.build_hierarchical_model(data)

        # 推定
        results = self.model_builder.fit(model)
        return results

class BayesianModelBuilder:
    """論文定式化に基づいた階層交互作用モデルの構築"""

    def __init__(self, config: 'Config'):
        self.config = config
        self.draws = config.mcmc_draws
        self.tune = config.mcmc_tune
        self.chains = config.mcmc_chains
        self.target_accept = config.target_accept
        self.logger = logging.getLogger(self.__class__.__name__)

    def build_hierarchical_model(self, data: Dict) -> pm.Model:
        """
        論文式: μ = α + β(t) + γ(r) + δ(s) + ζ(t,r) + η(r,s) + θ(t,s)
        """
        with pm.Model() as model:
            # --- 事前分布 (Weakly Informative Priors) [cite: 128, 142] ---
            mu_global = pm.Normal("mu_global", mu=3.0, sigma=1.0)

            # 分散パラメータ (Half-Normal) [cite: 136, 138]
            sigma_year = pm.HalfNormal("sigma_year", sigma=0.5)
            sigma_region = pm.HalfNormal("sigma_region", sigma=0.5)
            sigma_sector = pm.HalfNormal("sigma_sector", sigma=0.5)
            sigma_ry = pm.HalfNormal("sigma_ry", sigma=0.2)
            sigma_rs = pm.HalfNormal("sigma_rs", sigma=0.2)
            sigma_ys = pm.HalfNormal("sigma_ys", sigma=0.2)
            sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.5)

            # --- 主効果 (非中心化) ---
            mu_year_offset = pm.Normal("mu_year_offset", mu=0, sigma=1, shape=data['n_times'])
            mu_year = pm.Deterministic("mu_year", mu_year_offset * sigma_year)

            mu_region_offset = pm.Normal("mu_region_offset", mu=0, sigma=1, shape=data['n_regions'])
            mu_region = pm.Deterministic("mu_region", mu_region_offset * sigma_region)

            mu_sector_offset = pm.Normal("mu_sector_offset", mu=0, sigma=1, shape=data['n_sectors'])
            mu_sector = pm.Deterministic("mu_sector", mu_sector_offset * sigma_sector)

            # --- 交互作用効果 (非中心化) ---
            # mu_region_year (ζ): 地域特有の時系列トレンド
            mu_ry_offset = pm.Normal("mu_ry_offset", mu=0, sigma=1,
                                     shape=(data['n_regions'], data['n_times']))
            mu_region_year = pm.Deterministic("mu_region_year", mu_ry_offset * sigma_ry)

            # mu_region_sector (η): 地域特有の産業特性
            mu_rs_offset = pm.Normal("mu_rs_offset", mu=0, sigma=1,
                                     shape=(data['n_regions'], data['n_sectors']))
            mu_region_sector = pm.Deterministic("mu_region_sector", mu_rs_offset * sigma_rs)

            # mu_year_sector (θ): 全国的な産業別変化パターン
            mu_ys_offset = pm.Normal("mu_ys_offset", mu=0, sigma=1,
                                     shape=(data['n_times'], data['n_sectors']))
            mu_year_sector = pm.Deterministic("mu_year_sector", mu_ys_offset * sigma_ys)

            # 線形予測子 μ
            mu = (mu_global +
                  mu_year[data['time_idx']] +
                  mu_region[data['region_idx']] +
                  mu_sector[data['sector_idx']] +
                  mu_region_year[data['region_idx'], data['time_idx']] +
                  mu_region_sector[data['region_idx'], data['sector_idx']] +
                  mu_year_sector[data['time_idx'], data['sector_idx']])

            # 尤度 (正規分布) [cite: 120]
            y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_obs, observed=data['y'])

        return model

    def fit(self, model: pm.Model) -> 'BayesianModelResults':
        self.logger.info("Starting MCMC sampling (NUTS)...")
        with model:
            trace = pm.sample(
                draws=self.draws, tune=self.tune, chains=self.chains,
                target_accept=self.target_accept, random_seed=42,
                return_inferencedata=True
            )
        summary = az.summary(trace)
        # --- ここで指標を計算 ---
        try:
            waic = az.waic(trace).elpd_waic
            loo = az.loo(trace).elpd_loo
        except Exception as e:
            self.logger.warning(f"Could not compute WAIC/LOO: {e}")
            waic, loo = None, None

        return BayesianModelResults(
            trace=trace, model=model, summary=summary,
            waic=waic, loo=loo
        )
print("✓ Bayesian model classes defined")

## 可視化クラス

In [ ]:
# ===== 可視化クラス =====

class APLVisualizer:
    """APL分析結果の可視化クラス"""

    def __init__(self, save_dir: str, dpi: int = 300):
        self.save_dir = save_dir
        self.dpi = dpi
        self.logger = logging.getLogger(self.__class__.__name__)

    def plot_regional_transition(self, apl_results: Dict,
                                regions: List[str],
                                linkage_type: str = 'backward') -> None:
        """地域別APLの時系列推移をプロット"""
        years = sorted(list(apl_results.keys()))

        if not years:
            self.logger.warning("APLデータが空です")
            return

        plt.figure(figsize=(12, 8))

        key = 'back' if linkage_type == 'backward' else 'forw'

        for i, region in enumerate(regions):
            y_values = []
            for year in years:
                value = apl_results[year][key][i]
                y_values.append(value if not np.isnan(value) else None)

            if any(v is not None for v in y_values):
                plt.plot(years, y_values, marker='o', linewidth=2,
                        markersize=6, label=region)

        direction = "後方" if linkage_type == 'backward' else "前方"
        plt.xlabel('年次', fontsize=14)
        plt.ylabel(f'{direction}APL', fontsize=14)
        plt.title(f'地域別 {direction}APLの推移', fontsize=16)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()

        filename = f"regional_{linkage_type}_apl_transition.png"
        plt.savefig(os.path.join(self.save_dir, filename),
                   dpi=self.dpi, bbox_inches='tight')
        plt.show()
        plt.close()

        self.logger.info(f"Saved: {filename}")

    def plot_trajectory(self, apl_results: Dict, regions: List[str],
                       highlight_region: Optional[str] = None) -> None:
        """地域別APLの軌跡プロット（後方 vs 前方）"""
        years = sorted(list(apl_results.keys()))

        if not years:
            self.logger.warning("APLデータが空です")
            return

        plt.figure(figsize=(12, 10))
        colors = plt.cm.tab10(np.arange(len(regions)))

        for i, region in enumerate(regions):
            x_values = []
            y_values = []
            valid_years = []

            for year in years:
                back_val = apl_results[year]['back'][i]
                forw_val = apl_results[year]['forw'][i]

                if not np.isnan(back_val) and not np.isnan(forw_val):
                    x_values.append(back_val)
                    y_values.append(forw_val)
                    valid_years.append(year)

            if len(x_values) == 0:
                continue

            # 強調表示の設定
            is_highlight = (highlight_region and highlight_region in region)
            linestyle = '-' if is_highlight else '--'
            linewidth = 4 if is_highlight else 2
            alpha = 1.0 if is_highlight else 0.7
            markersize = 12 if is_highlight else 8

            if len(x_values) > 1:
                plt.plot(x_values, y_values, marker='o', label=region,
                        color=colors[i], linewidth=linewidth,
                        linestyle=linestyle, markersize=markersize, alpha=alpha)
            else:
                plt.scatter(x_values[0], y_values[0], marker='o', label=region,
                           color=colors[i], s=markersize**2, alpha=alpha)

            # アノテーション
            self._add_annotations(x_values, y_values, valid_years,
                                region, is_highlight)

        plt.xlabel('後方連関の長さ (APL 生産額加重平均)',
                  fontsize=14, fontweight='bold')
        plt.ylabel('前方連関の長さ (APL 生産額加重平均)',
                  fontsize=14, fontweight='bold')
        plt.title('地域別 APL位置の推移', fontsize=16, fontweight='bold', pad=20)
        plt.grid(True, alpha=0.4, linewidth=0.8)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()

        suffix = f"_{highlight_region}" if highlight_region else ""
        filename = f"regional_apl_trajectory{suffix}.png"
        plt.savefig(os.path.join(self.save_dir, filename),
                   dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved: {filename}")

    def _add_annotations(self, x_values: List[float], y_values: List[float],
                        years: List[str], region: str,
                        is_highlight: bool) -> None:
        """アノテーションを追加"""
        for j, year in enumerate(years):
            year_label = YearLabelConverter.to_japanese(year)

            if is_highlight:
                bbox_props = dict(boxstyle="round,pad=0.4", facecolor="yellow",
                                edgecolor='orange', alpha=0.9, linewidth=1.0)
                fontweight = 'bold'
                fontsize = 11
            else:
                bbox_props = dict(boxstyle="round,pad=0.3", facecolor="white",
                                edgecolor='lightgray', alpha=0.7, linewidth=0.6)
                fontweight = 'normal'
                fontsize = 10

            plt.annotate(f"{region}({year_label})", (x_values[j], y_values[j]),
                        xytext=(8, 8), textcoords='offset points',
                        fontsize=fontsize, fontweight=fontweight,
                        ha='left', va='bottom', bbox=bbox_props)

    def plot_bayesian_variance(self, results: BayesianModelResults):
        """資料(source: 510-541)の分散寄与分解を再現"""
        # 抽出する分散パラメータ
        sigma_vars = ['sigma_year', 'sigma_sector', 'sigma_ry', 'sigma_rs', 'sigma_ys']
        labels = ['年次', '産業', '地域×年次', '地域×産業', '年次×産業']
        data = []
        for var, label in zip(sigma_vars, labels):
            # 分散 (sigma^2) を計算
            samples = results.trace.posterior[var].values.flatten()**2
            data.append({'Effect': label, 'Mean': np.mean(samples),
                        'Lower': az.hdi(samples, hdi_prob=0.94)[0],
                        'Upper': az.hdi(samples, hdi_prob=0.94)[1]})

        df = pd.DataFrame(data)
        plt.figure(figsize=(10, 6))
        plt.bar(df['Effect'], df['Mean'], yerr=[df['Mean']-df['Lower'], df['Upper']-df['Mean']], capsize=5)
        plt.ylabel('分散 ($\sigma^2$)')
        plt.title('各階層効果の寄与分散')
        plt.show()

    def plot_caterpillar(self, results: BayesianModelResults, region_name: str, config: Config):
            """産業別交互作用プロット"""
            reg_idx = config.regions.index(region_name)
            samples = results.trace.posterior['mu_region_sector'][:, :, reg_idx, :].values.reshape(-1, len(config.sectors))

            # 事後平均でソート
            means = samples.mean(axis=0)
            sorted_idx = np.argsort(means)

            plt.figure(figsize=(8, 12))
            for i, idx in enumerate(sorted_idx):
                hdi = az.hdi(samples[:, idx], hdi_prob=0.94)
                plt.plot([hdi[0], hdi[1]], [i, i], color='navy', lw=2)
                plt.plot(means[idx], i, 'o', color='navy')

            plt.axvline(0, color='red', linestyle='--')
            plt.yticks(range(len(config.sectors)), [config.sectors[i] for i in sorted_idx])
            plt.title(f'{region_name}固有の産業別交互作用効果 (94%信用区間)')
            plt.show()

    def plot_deviation_trend(self, bayesian_results: 'BayesianModelResults', config: 'Config', target_region: 'str' = '四国'):
            """
            論文図6：全国トレンドからの逸脱度の時系列変化をプロット
            """
            reg_idx = config.regions.index(target_region)

            # 1. 和暦ラベルと西暦の対応マップ
            year_to_num = {'S60': 1985, 'H2': 1990, 'H7': 1995, 'H17': 2005}

            # 2. 文字列ソートではなく、西暦順に並び替えたリストを作成
            # これで ['S60', 'H2', 'H7', 'H17'] の順が保証される
            sorted_years_label = sorted(config.years.keys(), key=lambda y: year_to_num[y])
            num_years = [year_to_num[y] for y in sorted_years_label]

            plt.figure(figsize=(12, 8))
            colors = plt.cm.get_cmap('tab20', len(config.sectors))

            for s_idx, sector in enumerate(config.sectors):
                deviations = []
                for year_label in sorted_years_label:
                    # 3. データの取得も時系列順に行う
                    # インデックスを正しく特定するために year_label の元の位置を確認
                    original_idx = sorted(config.years.keys()).index(year_label) # データが入っている順番

                    # ベイズ結果の posterior は、恐らくデータの投入順(S60, H2, H7, H17)に並んでいるはずです
                    # data['time_idx'] のマッピングと一致させる必要があります
                    t_idx = list(config.years.keys()).index(year_label)

                    y_s = bayesian_results.trace.posterior['mu_year_sector'][:, :, t_idx, s_idx].mean().item()
                    r_s = bayesian_results.trace.posterior['mu_region_sector'][:, :, reg_idx, s_idx].mean().item()
                    deviations.append(y_s + r_s)

                # プロット
                plt.plot(num_years, deviations, marker='o', label=sector, color=colors(s_idx), alpha=0.7)

            # (以下、垂直線の処理は前回と同じ)
            events = {1988: 'S63 瀬戸大橋', 1998: 'H10 明石海峡大橋', 1999: 'H11 しまなみ海道'}
            for ev_y, label in events.items():
                plt.axvline(x=ev_y, color='red', linestyle='--', alpha=0.4)
                plt.text(ev_y, plt.ylim()[0], label, rotation=90, verticalalignment='bottom', color='red')

            plt.xticks(num_years, sorted_years_label) # 軸の見た目は和暦
            plt.title(f'{target_region}の全国トレンドからの逸脱度')
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    def plot_tertiary_correlation(self, apl_results: Dict, tertiary_ratios: Dict, config: 'Config', linkage_type: str = 'backward'):
        """
        論文図7：第三次産業比率とAPLの相関散布図をプロット
        """
        all_apl = []
        all_ratios = []
        labels = []

        key = 'back' if linkage_type == 'backward' else 'forw'

        for year in sorted(apl_results.keys()):
            for i, region in enumerate(config.regions):
                val = apl_results[year][key][i]
                ratio = tertiary_ratios[year][i]
                if not np.isnan(val) and not np.isnan(ratio):
                    all_apl.append(val)
                    all_ratios.append(ratio)
                    labels.append(f"{region}({year})")

        # 相関係数の計算
        r, p = pearsonr(all_ratios, all_apl)

        plt.figure(figsize=(10, 7))
        sns.regplot(x=all_ratios, y=all_apl, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})

        # 各点にラベルを付ける
        for i, txt in enumerate(labels):
            plt.annotate(txt, (all_ratios[i], all_apl[i]), fontsize=8, alpha=0.7)

        direction = "後方" if linkage_type == 'backward' else "前方"
        plt.title(f'第三次産業比率と{direction}APLの相関 (r={r:.3f}, p={p:.3f})', fontsize=15)
        plt.xlabel('第三次産業比率', fontsize=13)
        plt.ylabel(f'{direction}APL', fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.show()

    def plot_mcmc_trace(self, bayesian_results: BayesianModelResults,
                       linkage_type: str = 'backward',
                       params_to_plot: list = None):
        """
        MCMCのトレースプロットを作成

        Parameters
        ----------
        bayesian_results : BayesianModelResults
            ベイズ推定結果
        linkage_type : str
            'backward' または 'forward'
        params_to_plot : list, optional
            プロットするパラメータのリスト。Noneの場合は主要パラメータを自動選択
        """
        # デフォルトでプロットする主要パラメータ
        if params_to_plot is None:
            params_to_plot = [
                'mu_global',
                'sigma_year',
                'sigma_region',
                'sigma_sector',
                'sigma_ry',
                'sigma_rs',
                'sigma_ys',
                'sigma_obs'
            ]

        # トレースプロットの作成
        fig = az.plot_trace(
            bayesian_results.trace,
            var_names=params_to_plot,
            figsize=(14, len(params_to_plot) * 2),
            compact=False
        )

        direction = "後方" if linkage_type == 'backward' else "前方"
        plt.suptitle(f'MCMCトレースプロット ({direction}連関)',
                     fontsize=16, fontweight='bold', y=1.001)
        plt.tight_layout()

        # 保存
        filename = f'mcmc_trace_{linkage_type}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight')
        plt.show()
        plt.close()

        self.logger.info(f"Saved MCMC trace plot: {filename}")

    def plot_variance_contribution(self, results: BayesianModelResults,
                                  linkage_type: str = 'backward',
                                  target_region: str = '四国'):
        """
        各階層効果の分散寄与を棒グラフで可視化（論文図用）

        Parameters
        ----------
        results : BayesianModelResults
            ベイズ推定結果
        linkage_type : str
            'backward' または 'forward'
        target_region : str
            分析対象地域
        """
        direction = '後方' if linkage_type == 'backward' else '前方'

        # 分散パラメータの抽出（sigma^2を計算）
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        means = []
        lowers = []
        uppers = []
        labels = []

        for label, param_name in variance_components.items():
            # sigma^2 を計算（分散）
            samples = results.trace.posterior[param_name].values.flatten() ** 2
            mean_val = np.mean(samples)
            hdi = az.hdi(samples, hdi_prob=0.95)

            means.append(mean_val)
            lowers.append(mean_val - hdi[0])
            uppers.append(hdi[1] - mean_val)
            labels.append(label)

        # プロット
        fig, ax = plt.subplots(figsize=(8, 6))

        # 色分け（主効果 vs 交互作用）
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        x_pos = np.arange(len(labels))
        bars = ax.bar(x_pos, means,
                      yerr=[lowers, uppers],
                      capsize=5,
                      color=colors,
                      alpha=0.8,
                      edgecolor='black',
                      linewidth=1.2)

        # 軸設定
        ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=0, fontsize=12)
        ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                    f'平均値±95%信用区間',
                    fontsize=14, fontweight='bold', pad=15)

        # グリッド
        ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)

        # y軸の範囲を調整
        ax.set_ylim(0, max(means) * 1.3)

        plt.tight_layout()

        # 保存
        filename = f'variance_contribution_{linkage_type}_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance contribution plot: {filename}")


    def plot_variance_comparison(self, results_back: BayesianModelResults,
                                results_forw: BayesianModelResults,
                                target_region: str = '四国'):
        """
        後方APLと前方APLの分散寄与を並べて比較（2パネル図）

        Parameters
        ----------
        results_back : BayesianModelResults
            後方連関のベイズ推定結果
        results_forw : BayesianModelResults
            前方連関のベイズ推定結果
        target_region : str
            分析対象地域
        """
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        for idx, (results, direction, ax) in enumerate([
            (results_back, '後方', axes[0]),
            (results_forw, '前方', axes[1])
        ]):
            means = []
            lowers = []
            uppers = []
            labels = []

            for label, param_name in variance_components.items():
                samples = results.trace.posterior[param_name].values.flatten() ** 2
                mean_val = np.mean(samples)
                hdi = az.hdi(samples, hdi_prob=0.95)

                means.append(mean_val)
                lowers.append(mean_val - hdi[0])
                uppers.append(hdi[1] - mean_val)
                labels.append(label)

            # プロット
            x_pos = np.arange(len(labels))
            ax.bar(x_pos, means,
                  yerr=[lowers, uppers],
                  capsize=5,
                  color=colors,
                  alpha=0.8,
                  edgecolor='black',
                  linewidth=1.2)

            # 軸設定
            ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=13, fontweight='bold')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(labels, rotation=0, fontsize=11)
            ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                        f'平均値±95%信用区間',
                        fontsize=13, fontweight='bold', pad=12)
            ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
            ax.set_axisbelow(True)

            # y軸の範囲を統一
            if idx == 0:
                y_max_back = max(means) * 1.3
            else:
                y_max_forw = max(means) * 1.3

        # y軸の範囲を両方のパネルで統一
        y_max_common = max(y_max_back, y_max_forw)
        axes[0].set_ylim(0, y_max_common)
        axes[1].set_ylim(0, y_max_common)

        plt.tight_layout()

        # 保存
        filename = f'variance_comparison_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance comparison plot: {filename}")


    def plot_variance_contribution(self, results: BayesianModelResults,
                                  linkage_type: str = 'backward',
                                  target_region: str = '四国'):
        """
        各階層効果の分散寄与を棒グラフで可視化（論文図用）

        Parameters
        ----------
        results : BayesianModelResults
            ベイズ推定結果
        linkage_type : str
            'backward' または 'forward'
        target_region : str
            分析対象地域
        """
        direction = '後方' if linkage_type == 'backward' else '前方'

        # 分散パラメータの抽出（sigma^2を計算）
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        means = []
        lowers = []
        uppers = []
        labels = []

        for label, param_name in variance_components.items():
            # sigma^2 を計算（分散）
            samples = results.trace.posterior[param_name].values.flatten() ** 2
            mean_val = np.mean(samples)
            hdi = az.hdi(samples, hdi_prob=0.95)

            means.append(mean_val)
            lowers.append(mean_val - hdi[0])
            uppers.append(hdi[1] - mean_val)
            labels.append(label)

        # プロット
        fig, ax = plt.subplots(figsize=(8, 6))

        # 色分け（主効果 vs 交互作用）
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        x_pos = np.arange(len(labels))
        bars = ax.bar(x_pos, means,
                      yerr=[lowers, uppers],
                      capsize=5,
                      color=colors,
                      alpha=0.8,
                      edgecolor='black',
                      linewidth=1.2)

        # 軸設定
        ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=0, fontsize=12)
        ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                    f'平均値±95%信用区間',
                    fontsize=14, fontweight='bold', pad=15)

        # グリッド
        ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)

        # y軸の範囲を調整
        ax.set_ylim(0, max(means) * 1.3)

        plt.tight_layout()

        # 保存
        filename = f'variance_contribution_{linkage_type}_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance contribution plot: {filename}")


    def plot_variance_comparison(self, results_back: BayesianModelResults,
                                results_forw: BayesianModelResults,
                                target_region: str = '四国'):
        """
        後方APLと前方APLの分散寄与を並べて比較（2パネル図）

        Parameters
        ----------
        results_back : BayesianModelResults
            後方連関のベイズ推定結果
        results_forw : BayesianModelResults
            前方連関のベイズ推定結果
        target_region : str
            分析対象地域
        """
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        for idx, (results, direction, ax) in enumerate([
            (results_back, '後方', axes[0]),
            (results_forw, '前方', axes[1])
        ]):
            means = []
            lowers = []
            uppers = []
            labels = []

            for label, param_name in variance_components.items():
                samples = results.trace.posterior[param_name].values.flatten() ** 2
                mean_val = np.mean(samples)
                hdi = az.hdi(samples, hdi_prob=0.95)

                means.append(mean_val)
                lowers.append(mean_val - hdi[0])
                uppers.append(hdi[1] - mean_val)
                labels.append(label)

            # プロット
            x_pos = np.arange(len(labels))
            ax.bar(x_pos, means,
                  yerr=[lowers, uppers],
                  capsize=5,
                  color=colors,
                  alpha=0.8,
                  edgecolor='black',
                  linewidth=1.2)

            # 軸設定
            ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=13, fontweight='bold')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(labels, rotation=0, fontsize=11)
            ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                        f'平均値±95%信用区間',
                        fontsize=13, fontweight='bold', pad=12)
            ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
            ax.set_axisbelow(True)

            # y軸の範囲を統一
            if idx == 0:
                y_max_back = max(means) * 1.3
            else:
                y_max_forw = max(means) * 1.3

        # y軸の範囲を両方のパネルで統一
        y_max_common = max(y_max_back, y_max_forw)
        axes[0].set_ylim(0, y_max_common)
        axes[1].set_ylim(0, y_max_common)

        plt.tight_layout()

        # 保存
        filename = f'variance_comparison_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance comparison plot: {filename}")
    def plot_model_graph(self, save_name: str = 'bayesian_model_detailed.png'):
        """
        ベイズモデルの階層構造図（Plate Notation）を作成

        Parameters
        ----------
        save_name : str
            保存するファイル名
        """
        try:
            from graphviz import Digraph
        except ImportError:
            print("graphvizのインストールが必要です:")
            print("!apt-get install -y graphviz")
            print("!pip install graphviz")
            return

        # グラフの作成
        dot = Digraph(comment='Hierarchical Bayesian Model for APL Analysis')
        dot.attr(rankdir='TB', bgcolor='white')
        dot.attr('node', shape='ellipse', style='filled', fillcolor='lightblue',
                 fontname='sans-serif', fontsize='11')

        # ハイパーパラメータ（事前分布）
        dot.node('mu_global', 'mu_global\nNormal', fillcolor='lightgray')

        # 分散パラメータ（HalfNormal）
        variance_params = [
            ('sigma_year', 'sigma_year\nHalfNormal'),
            ('sigma_region', 'sigma_region\nHalfNormal'),
            ('sigma_sector', 'sigma_sector\nHalfNormal'),
            ('sigma_ry', 'sigma_ry\nHalfNormal'),
            ('sigma_rs', 'sigma_rs\nHalfNormal'),
            ('sigma_ys', 'sigma_ys\nHalfNormal'),
            ('sigma_obs', 'sigma_obs\nHalfNormal')
        ]

        for node_id, label in variance_params:
            dot.node(node_id, label, fillcolor='lightyellow')

        # 主効果パラメータ（Deterministic）
        main_effects = [
            ('mu_year', 'mu_year\nDeterministic', 'sigma_year', '4'),
            ('mu_region', 'mu_region\nDeterministic', 'sigma_region', '9'),
            ('mu_sector', 'mu_sector\nDeterministic', 'sigma_sector', '31')
        ]

        for node_id, label, sigma, dim in main_effects:
            dot.node(node_id, f'{label}\n[{dim}]', fillcolor='lightgreen')
            dot.edge(sigma, node_id)

        # 交互作用パラメータ（Deterministic）
        interactions = [
            ('mu_ry', 'mu_region_year\nDeterministic', 'sigma_ry', '9 x 4'),
            ('mu_rs', 'mu_region_sector\nDeterministic', 'sigma_rs', '9 x 31'),
            ('mu_ys', 'mu_year_sector\nDeterministic', 'sigma_ys', '4 x 31')
        ]

        for node_id, label, sigma, dim in interactions:
            dot.node(node_id, f'{label}\n[{dim}]', fillcolor='lightcoral')
            dot.edge(sigma, node_id)

        # 線形予測子（観測されない）
        dot.node('mu_pred', 'μ_pred\nNormal', fillcolor='white', shape='box')

        # 観測データ
        dot.node('y_obs', 'y_obs\nNormal\n[1116]', fillcolor='lightsteelblue',
                 style='filled,bold', penwidth='2')

        # エッジの追加（全体構造）
        dot.edge('mu_global', 'mu_pred')
        dot.edge('mu_year', 'mu_pred')
        dot.edge('mu_region', 'mu_pred')
        dot.edge('mu_sector', 'mu_pred')
        dot.edge('mu_ry', 'mu_pred')
        dot.edge('mu_rs', 'mu_pred')
        dot.edge('mu_ys', 'mu_pred')
        dot.edge('mu_pred', 'y_obs')
        dot.edge('sigma_obs', 'y_obs')

        # レンダリングと保存
        output_path = os.path.join(self.save_dir, save_name.replace('.png', ''))
        dot.render(output_path, format='png', cleanup=True)

        # 画像表示
        from IPython.display import Image, display
        display(Image(filename=f'{output_path}.png'))

        self.logger.info(f"Saved model graph: {save_name}")
        print(f"✓ モデル図を保存しました: {output_path}.png")

    def plot_model_graph_simple(self, save_name: str = 'bayesian_model_simple.png'):
        """
        シンプルなモデル構造図を作成（論文掲載用）
        """
        try:
            from graphviz import Digraph
        except ImportError:
            print("graphvizが必要です: !pip install graphviz")
            return

        dot = Digraph(comment='APL Bayesian Model Structure')
        dot.attr(rankdir='LR', bgcolor='white', size='10,6')
        dot.attr('node', shape='ellipse', fontname='sans-serif', fontsize='12')

        # 階層レベル1: ハイパーパラメータ
        with dot.subgraph(name='cluster_0') as c:
            c.attr(label='事前分布', fontsize='14', style='dashed')
            c.node('hyper', 'μ₀, σ₀', fillcolor='lightgray', style='filled')

        # 階層レベル2: パラメータ
        with dot.subgraph(name='cluster_1') as c:
            c.attr(label='階層パラメータ', fontsize='14', style='dashed')
            c.node('year', '年次効果\nβ(t)', fillcolor='lightblue', style='filled')
            c.node('region', '地域効果\nγ(r)', fillcolor='lightblue', style='filled')
            c.node('sector', '産業効果\nδ(s)', fillcolor='lightblue', style='filled')
            c.node('interact', '交互作用\nζ,η,θ', fillcolor='lightcoral', style='filled')

        # 階層レベル3: 観測モデル
        with dot.subgraph(name='cluster_2') as c:
            c.attr(label='観測モデル', fontsize='14', style='dashed')
            c.node('mu', 'μ = α + β + γ + δ + ...',
                   fillcolor='lightyellow', style='filled', shape='box')
            c.node('y', 'APL観測値\ny ~ N(μ, σ)',
                   fillcolor='lightsteelblue', style='filled,bold', penwidth='2')

        # エッジ
        dot.edge('hyper', 'year')
        dot.edge('hyper', 'region')
        dot.edge('hyper', 'sector')
        dot.edge('hyper', 'interact')
        dot.edge('year', 'mu')
        dot.edge('region', 'mu')
        dot.edge('sector', 'mu')
        dot.edge('interact', 'mu')
        dot.edge('mu', 'y')

        # 保存
        output_path = os.path.join(self.save_dir, save_name.replace('.png', ''))
        dot.render(output_path, format='png', cleanup=True)

        from IPython.display import Image, display
        display(Image(filename=f'{output_path}.png'))

        self.logger.info(f"Saved simple model graph: {save_name}")
        print(f"✓ シンプルなモデル図を保存しました: {output_path}.png")

    def plot_variance_contribution(self, results: BayesianModelResults,
                                   linkage_type: str = 'backward',
                                   target_region: str = '四国'):
        """
        各階層効果の分散寄与を棒グラフで可視化（論文図用）

        Parameters
        ----------
        results : BayesianModelResults
            ベイズ推定結果
        linkage_type : str
            'backward' または 'forward'
        target_region : str
            分析対象地域
        """
        direction = '後方' if linkage_type == 'backward' else '前方'

        # 分散パラメータの抽出（sigma^2を計算）
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        means = []
        lowers = []
        uppers = []
        labels = []

        for label, param_name in variance_components.items():
            # sigma^2 を計算（分散）
            samples = results.trace.posterior[param_name].values.flatten() ** 2
            mean_val = np.mean(samples)
            hdi = az.hdi(samples, hdi_prob=0.95)

            means.append(mean_val)
            lowers.append(mean_val - hdi[0])
            uppers.append(hdi[1] - mean_val)
            labels.append(label)

        # プロット
        fig, ax = plt.subplots(figsize=(8, 6))

        # 色分け（主効果 vs 交互作用）
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        x_pos = np.arange(len(labels))
        bars = ax.bar(x_pos, means,
                      yerr=[lowers, uppers],
                      capsize=5,
                      color=colors,
                      alpha=0.8,
                      edgecolor='black',
                      linewidth=1.2)

        # 軸設定
        ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=0, fontsize=12)
        ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                     f'平均値±95%信用区間',
                     fontsize=14, fontweight='bold', pad=15)

        # グリッド
        ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)

        # y軸の範囲を調整
        ax.set_ylim(0, max(means) * 1.3)

        plt.tight_layout()

        # 保存
        filename = f'variance_contribution_{linkage_type}_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance contribution plot: {filename}")

    def plot_variance_comparison(self, results_back: BayesianModelResults,
                                 results_forw: BayesianModelResults,
                                 target_region: str = '四国'):
        """
        後方APLと前方APLの分散寄与を並べて比較（2パネル図）

        Parameters
        ----------
        results_back : BayesianModelResults
            後方連関のベイズ推定結果
        results_forw : BayesianModelResults
            前方連関のベイズ推定結果
        target_region : str
            分析対象地域
        """
        variance_components = {
            '年次': 'sigma_year',
            '産業': 'sigma_sector',
            '地域×年次': 'sigma_ry',
            '地域×産業': 'sigma_rs',
            '年次×産業': 'sigma_ys'
        }

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        colors = ['#7BA8D1', '#E8A87C', '#D3D3D3', '#F4A6A6', '#A6C8E8']

        for idx, (results, direction, ax) in enumerate([
            (results_back, '後方', axes[0]),
            (results_forw, '前方', axes[1])
        ]):
            means = []
            lowers = []
            uppers = []
            labels = []

            for label, param_name in variance_components.items():
                samples = results.trace.posterior[param_name].values.flatten() ** 2
                mean_val = np.mean(samples)
                hdi = az.hdi(samples, hdi_prob=0.95)

                means.append(mean_val)
                lowers.append(mean_val - hdi[0])
                uppers.append(hdi[1] - mean_val)
                labels.append(label)

            # プロット
            x_pos = np.arange(len(labels))
            ax.bar(x_pos, means,
                   yerr=[lowers, uppers],
                   capsize=5,
                   color=colors,
                   alpha=0.8,
                   edgecolor='black',
                   linewidth=1.2)

            # 軸設定
            ax.set_ylabel('分散 ($\\sigma^2$)', fontsize=13, fontweight='bold')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(labels, rotation=0, fontsize=11)
            ax.set_title(f'{target_region}における各階層効果の寄与分散 ({direction}APL)\n'
                         f'平均値±95%信用区間',
                         fontsize=13, fontweight='bold', pad=12)
            ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
            ax.set_axisbelow(True)

            # y軸の範囲を統一
            if idx == 0:
                y_max_back = max(means) * 1.3
            else:
                y_max_forw = max(means) * 1.3

        # y軸の範囲を両方のパネルで統一
        y_max_common = max(y_max_back, y_max_forw)
        axes[0].set_ylim(0, y_max_common)
        axes[1].set_ylim(0, y_max_common)

        plt.tight_layout()

        # 保存
        filename = f'variance_comparison_{target_region}.png'
        filepath = os.path.join(self.save_dir, filename)
        plt.savefig(filepath, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close()

        self.logger.info(f"Saved variance comparison plot: {filename}")

    def plot_heatmap(matrix: np.ndarray, regions: List[str], year: str,
                    linkage_type: str = 'forward', save_dir: str = './apl_figs/',
                    cmap: str = 'Greys', dpi: int = 300) -> None:
        """地域間APL行列のヒートマップ"""
        plt.figure(figsize=(8, 6))

        sns.heatmap(matrix, xticklabels=regions, yticklabels=regions,
                   annot=True, fmt=".2f", cmap=cmap,
                   cbar_kws={'label': 'APL'})

        direction = "前方" if linkage_type == 'forward' else "後方"
        plt.title(f"地域間{direction}APL行列（{year}）")
        plt.xlabel("地域（行き先）" if linkage_type == 'forward'
                  else "供給地（生産側）")
        plt.ylabel("地域（出発地）" if linkage_type == 'forward'
                  else "需要地（最終需要側）")
        plt.tight_layout()

        filename = f"region_apl_matrix_{linkage_type}_{year}.png"
        plt.savefig(os.path.join(save_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.show()
        plt.close()

print("✓ Visualizer class (part 1) defined")

In [ ]:
# ===== 表出力クラス =====
class TableExporter:
    """表の出力クラス"""

    def __init__(self, save_dir: str):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.logger = logging.getLogger(self.__class__.__name__)

    def create_apl_table(self, apl_results: Dict,
                        regions: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """地域別APL表を作成"""
        years = sorted(list(apl_results.keys()))

        if not years:
            self.logger.warning("APLデータが空です")
            return None, None

        # 後方APL
        back_data = {
            region: [apl_results[year]['back'][i] for year in years]
            for i, region in enumerate(regions)
        }

        # 前方APL
        forw_data = {
            region: [apl_results[year]['forw'][i] for year in years]
            for i, region in enumerate(regions)
        }

        back_df = pd.DataFrame(back_data, index=years)
        forw_df = pd.DataFrame(forw_data, index=years)

        # CSV保存
        back_df.to_csv(os.path.join(self.save_dir, "regional_back_apl.csv"),
                      encoding='utf-8-sig')
        forw_df.to_csv(os.path.join(self.save_dir, "regional_forw_apl.csv"),
                      encoding='utf-8-sig')

        # 表示
        print("\n=== 後方APL（地域別）===")
        print(back_df.round(3))
        print("\n=== 前方APL（地域別）===")
        print(forw_df.round(3))

        self.logger.info("APL tables saved")

        return back_df, forw_df

    def save_matrices_to_excel(self, A: np.ndarray, L: np.ndarray,
                              labels: List[str], year: str) -> None:
        """投入係数行列とレオンチェフ逆行列をExcelに保存"""
        filename = os.path.join(self.save_dir, f"matrix_{year}.xlsx")

        with pd.ExcelWriter(filename) as writer:
            pd.DataFrame(A, index=labels, columns=labels).to_excel(
                writer, sheet_name='投入係数A')
            pd.DataFrame(L, index=labels, columns=labels).to_excel(
                writer, sheet_name='逆行列係数L')

        self.logger.info(f"Saved matrices: matrix_{year}.xlsx")

print("✓ Visualizer class (part 2) and table exporter defined")

class AdditionalTableExporter:
    """研究発表で使用する追加の表を出力"""

    def __init__(self, table_dir: str):
        self.table_dir = table_dir

    def export_correlation_table(self, apl_results: Dict,
                                 tertiary_ratios: Dict,
                                 config: Config) -> None:
        """第三次産業比率とAPLの相関係数表を出力"""

        # データ準備
        all_back_apl = []
        all_forw_apl = []
        all_tertiary = []

        for year in sorted(apl_results.keys()):
            for i, region in enumerate(config.regions):
                all_back_apl.append(apl_results[year]['back'][i])
                all_forw_apl.append(apl_results[year]['forw'][i])
                all_tertiary.append(tertiary_ratios[year][i])

        # 相関係数計算
        from scipy.stats import pearsonr

        corr_back, p_back = pearsonr(all_tertiary, all_back_apl)
        corr_forw, p_forw = pearsonr(all_tertiary, all_forw_apl)

        # 表作成
        corr_df = pd.DataFrame({
            '指標': ['後方APL', '前方APL'],
            '相関係数 (r)': [f'{corr_back:.3f}', f'{corr_forw:.3f}'],
            'p値': [f'{p_back:.3f}', f'{p_forw:.3f}'],
            '有意性': ['有意' if p_back < 0.05 else '非有意',
                      '有意' if p_forw < 0.05 else '非有意']
        })

        # 保存
        filename = 'correlation_tertiary_apl.csv'
        corr_df.to_csv(os.path.join(self.table_dir, filename),
                      index=False, encoding='utf-8-sig')

        print(f"✓ 相関係数表を保存: {filename}")
        print(corr_df.to_string(index=False))

    def export_shikoku_year_heatmap_table(self,
                                          bayesian_results: 'BayesianModelResults',
                                          config: Config) -> None:
        """四国×年次交互作用のヒートマップ表を出力"""

        # 四国のインデックスを取得
        shikoku_idx = config.regions.index('四国')

        # 交互作用効果を抽出
        years = sorted(config.years.keys())
        year_effects = []

        for year_idx, year in enumerate(years):
            # 地域×年次交互作用の平均値
            effect = bayesian_results.trace.posterior['mu_region_year'][
                :, :, shikoku_idx, year_idx
            ].mean().item()
            year_effects.append(effect)

        # 表作成
        heatmap_df = pd.DataFrame({
            '年次': years,
            '四国×年次交互作用': [f'{e:.3f}' for e in year_effects]
        })

        # 保存
        filename = 'shikoku_year_interaction.csv'
        heatmap_df.to_csv(os.path.join(self.table_dir, filename),
                         index=False, encoding='utf-8-sig')

        print(f"✓ 四国×年次交互作用表を保存: {filename}")
        print(heatmap_df.to_string(index=False))

    def export_deviation_from_trend(self,
                                   bayesian_results: 'BayesianModelResults',
                                   config: Config,
                                   target_region: str = '四国') -> None:
        """全国トレンドからの逸脱度（時系列）表を出力"""

        region_idx = config.regions.index(target_region)
        years = sorted(config.years.keys())

        # 産業別の逸脱度を計算
        deviation_data = []

        for sector_idx, sector in enumerate(config.sectors):
            sector_deviations = []

            for year_idx in range(len(years)):
                # 年次×産業交互作用
                year_sector = bayesian_results.trace.posterior['mu_year_sector'][
                    :, :, year_idx, sector_idx
                ].mean().item()

                # 地域×産業交互作用
                region_sector = bayesian_results.trace.posterior['mu_region_sector'][
                    :, :, region_idx, sector_idx
                ].mean().item()

                # 合計＝逸脱度
                deviation = year_sector + region_sector
                sector_deviations.append(deviation)

            row = {'産業': sector}
            for year, dev in zip(years, sector_deviations):
                row[year] = f'{dev:.3f}'

            deviation_data.append(row)

        # 表作成
        deviation_df = pd.DataFrame(deviation_data)

        # 保存
        filename = f'{target_region}_deviation_from_trend.csv'
        deviation_df.to_csv(os.path.join(self.table_dir, filename),
                           index=False, encoding='utf-8-sig')

        print(f"✓ {target_region}の全国トレンドからの逸脱度表を保存: {filename}")
        print(deviation_df.head(10).to_string(index=False))

## パイプライン・実行クラス

In [ ]:
# ===== メインパイプラインクラス =====

class APLAnalysisPipeline:
    """APL分析パイプライン"""

    def __init__(self, config: Config):
        self.config = config
        self.loader = None  # 産業分類取得後に初期化
        self.calculator = APLCalculator()
        self.aggregator = RegionalAggregator()
        self.visualizer = APLVisualizer(config.save_dir, config.figure_dpi)
        self.table_exporter = TableExporter(config.table_dir)
        self.results = AnalysisResults()
        self.logger = logging.getLogger(self.__class__.__name__)

    def run_full_analysis(self) -> AnalysisResults:
        """完全な分析パイプラインを実行"""
        self.logger.info("=" * 60)
        self.logger.info("APL Analysis Pipeline Started")
        self.logger.info("=" * 60)

        # フェーズ1: データ読み込みとAPL計算
        self._load_and_calculate()

        # フェーズ2: 可視化
        self._create_visualizations()

        # フェーズ3: 表の出力
        self._export_tables()

        self.logger.info("=" * 60)
        self.logger.info("APL Analysis Pipeline Completed")
        self.logger.info("=" * 60)

        return self.results

    def _load_and_calculate(self) -> None:
        """データ読み込みとAPL計算"""
        self.logger.info("\n--- Phase 1: Data Loading and APL Calculation ---")

        for year, filepath in self.config.years.items():
            try:
                self.logger.info(f"\nProcessing {year}...")

                # データ読み込み
                io_data = self.loader.load(filepath, year)
                self.results.io_data[year] = io_data

                # APL計算
                linkage_results = self.calculator.calculate(
                    io_data.transaction_matrix, io_data.gross_output
                )
                self.results.raw_linkages[year] = linkage_results

                # 各種集計
                self._aggregate_regional_apl(year, io_data, linkage_results)
                self._aggregate_sectoral_apl(year, io_data, linkage_results)
                self._aggregate_shikoku_sectoral_apl(year, io_data, linkage_results)
                self._calculate_tertiary_ratio(year, io_data)

                # 行列保存
                self.table_exporter.save_matrices_to_excel(
                    linkage_results.input_coefficient,
                    linkage_results.leontief_inverse,
                    io_data.labels, year
                )

                self.logger.info(f"✓ {year} completed")

            except Exception as e:
                self.logger.error(f"✗ Error in {year}: {e}", exc_info=True)
                continue

    def _aggregate_regional_apl(self, year: str, io_data: IOTableData,
                               linkage_results: LinkageResults) -> None:
        """地域別APLを集計"""
        apl_matrix = linkage_results.apl_matrix

        back_apl_raw = np.nanmean(apl_matrix, axis=0)
        back_regional = self.aggregator.aggregate_by_region(
            back_apl_raw, io_data.labels, io_data.gross_output, self.config.regions
        )

        forw_apl_raw = np.nanmean(apl_matrix, axis=1)
        forw_regional = self.aggregator.aggregate_by_region(
            forw_apl_raw, io_data.labels, io_data.gross_output, self.config.regions
        )

        self.results.apl_results[year] = {'back': back_regional, 'forw': forw_regional}

    def _aggregate_sectoral_apl(self, year: str, io_data: IOTableData,
                               linkage_results: LinkageResults) -> None:
        """産業別APLを集計"""
        apl_matrix = linkage_results.apl_matrix

        back_apl_raw = np.nanmean(apl_matrix, axis=0)
        back_sectoral = self.aggregator.aggregate_by_sector(
            back_apl_raw, io_data.labels, io_data.gross_output, self.config.sectors
        )

        forw_apl_raw = np.nanmean(apl_matrix, axis=1)
        forw_sectoral = self.aggregator.aggregate_by_sector(
            forw_apl_raw, io_data.labels, io_data.gross_output, self.config.sectors
        )

        self.results.sector_apl_results[year] = {
            'back': back_sectoral, 'forw': forw_sectoral
        }

    def _aggregate_shikoku_sectoral_apl(self, year: str, io_data: IOTableData,
                                       linkage_results: LinkageResults) -> None:
        """四国の産業別APLを集計"""
        apl_matrix = linkage_results.apl_matrix
        back_apl_raw = np.nanmean(apl_matrix, axis=0)
        forw_apl_raw = np.nanmean(apl_matrix, axis=1)

        shikoku_back = []
        shikoku_forw = []

        for sector in self.config.sectors:
            target_label = f"四国_{sector}"
            try:
                idx = io_data.labels.index(target_label)
                shikoku_back.append(back_apl_raw[idx])
                shikoku_forw.append(forw_apl_raw[idx])
            except ValueError:
                shikoku_back.append(np.nan)
                shikoku_forw.append(np.nan)

        self.results.shikoku_sector_apl[year] = {
            'back': shikoku_back, 'forw': shikoku_forw
        }

    def _calculate_tertiary_ratio(self, year: str, io_data: IOTableData) -> None:
        """地域ごとの第三次産業比率を計算"""
        tertiary_ratio_year = []

        for region in self.config.regions:
            total_output = 0
            tertiary_output = 0

            for label, output_val in io_data.gross_output.items():
                try:
                    r, s = label.split('_', 1)
                    if r == region:
                        total_output += output_val
                        if s in self.config.tertiary_sectors:
                            tertiary_output += output_val
                except ValueError:
                    continue

            ratio = tertiary_output / total_output if total_output > 0 else np.nan
            tertiary_ratio_year.append(ratio)

        self.results.tertiary_ratios[year] = tertiary_ratio_year

    def _create_visualizations(self) -> None:
        """可視化を作成"""
        self.logger.info("\n--- Phase 2: Visualization ---")

        self.logger.info("Creating regional transition plots...")
        self.visualizer.plot_regional_transition(
            self.results.apl_results, self.config.regions, 'backward'
        )
        self.visualizer.plot_regional_transition(
            self.results.apl_results, self.config.regions, 'forward'
        )

        self.logger.info("Creating trajectory plots...")
        self.visualizer.plot_trajectory(
            self.results.apl_results, self.config.regions, None
        )
        self.visualizer.plot_trajectory(
            self.results.apl_results, self.config.regions, '四国'
        )

        self.logger.info("✓ Visualizations completed")

    def _export_tables(self) -> None:
        """表を出力"""
        self.logger.info("\n--- Phase 3: Table Export ---")
        self.table_exporter.create_apl_table(
            self.results.apl_results, self.config.regions
        )
        self.logger.info("✓ Tables exported")

    def run_bayesian_analysis(self, linkage_type: str = 'backward', auto_save: bool = True) -> 'BayesianModelResults':
            """ベイズ分析を実行（論文モデルを直接呼び出す）"""
            self.logger.info("\n--- Bayesian Analysis (Paper-based Model) ---")

            # 1. Builderの初期化
            model_builder = BayesianModelBuilder(self.config)
            # 2. Analyzerの初期化
            analyzer = RegionalAPLAnalyzer(model_builder, self.config)
            # 3. 分析実行
            results = analyzer.analyze(self.results, linkage_type)

            # 保存機能がある場合は自動保存（APLAnalysisPipelineWithSaveでオーバーライドされる想定）
            if auto_save and hasattr(self, 'bayesian_serializer'):
                self.logger.info("\n--- Auto-saving Bayesian results ---")
                self.bayesian_serializer.save_results(results, linkage_type)

            self.logger.info(f"✓ Bayesian analysis ({linkage_type}) completed")
            return results

print("✓ Pipeline class defined")

In [ ]:
# ===== 結果保存クラス =====

import pickle
from datetime import datetime
from typing import Optional

class ResultsSerializer:
    """分析結果の保存・読み込みクラス"""

    def __init__(self, save_dir: str = './apl_results/'):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.logger = logging.getLogger(self.__class__.__name__)

    def save_results(self, results: AnalysisResults,
                    filename: Optional[str] = None) -> str:
        """
        分析結果をpickleファイルに保存

        Parameters
        ----------
        results : AnalysisResults
            保存する分析結果
        filename : Optional[str]
            保存ファイル名（Noneの場合はタイムスタンプ付き）

        Returns
        -------
        str
            保存したファイルのパス
        """
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"apl_results_{timestamp}.pkl"

        filepath = os.path.join(self.save_dir, filename)

        try:
            with open(filepath, 'wb') as f:
                pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

            file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
            self.logger.info(f"✓ Results saved: {filepath} ({file_size:.2f} MB)")
            print(f"✓ 結果を保存しました: {filepath}")
            print(f"  ファイルサイズ: {file_size:.2f} MB")

            return filepath

        except Exception as e:
            self.logger.error(f"Failed to save results: {e}")
            raise

    def load_results(self, filename: str) -> AnalysisResults:
        """
        pickleファイルから分析結果を読み込み

        Parameters
        ----------
        filename : str
            読み込むファイル名またはパス

        Returns
        -------
        AnalysisResults
            読み込んだ分析結果
        """
        # ファイル名だけの場合はディレクトリと結合
        if not os.path.dirname(filename):
            filepath = os.path.join(self.save_dir, filename)
        else:
            filepath = filename

        try:
            with open(filepath, 'rb') as f:
                results = pickle.load(f)

            self.logger.info(f"✓ Results loaded: {filepath}")
            print(f"✓ 結果を読み込みました: {filepath}")

            # 読み込んだデータの内容を確認
            n_years = len(results.apl_results)
            print(f"  年次数: {n_years}")
            if n_years > 0:
                years = sorted(results.apl_results.keys())
                print(f"  期間: {years[0]} - {years[-1]}")

            return results

        except FileNotFoundError:
            self.logger.error(f"File not found: {filepath}")
            print(f"✗ ファイルが見つかりません: {filepath}")
            raise
        except Exception as e:
            self.logger.error(f"Failed to load results: {e}")
            raise

    def list_saved_results(self) -> List[str]:
        """
        保存されている結果ファイルの一覧を表示

        Returns
        -------
        List[str]
            保存済みファイルのリスト
        """
        pkl_files = [f for f in os.listdir(self.save_dir) if f.endswith('.pkl')]

        if not pkl_files:
            print("保存されている結果ファイルはありません")
            return []

        print(f"\n=== 保存済み結果ファイル ({len(pkl_files)}件) ===")
        for i, filename in enumerate(sorted(pkl_files), 1):
            filepath = os.path.join(self.save_dir, filename)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            mtime = datetime.fromtimestamp(os.path.getmtime(filepath))
            print(f"{i}. {filename}")
            print(f"   サイズ: {size_mb:.2f} MB, 更新日時: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")

        return pkl_files

    def save_specific_data(self, data: Any, name: str,
                          description: str = "") -> str:
        """
        特定のデータを個別に保存

        Parameters
        ----------
        data : Any
            保存するデータ（辞書、配列、データフレームなど）
        name : str
            データ名（ファイル名に使用）
        description : str
            データの説明（オプション）

        Returns
        -------
        str
            保存したファイルのパス
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{name}_{timestamp}.pkl"
        filepath = os.path.join(self.save_dir, filename)

        save_dict = {
            'data': data,
            'name': name,
            'description': description,
            'timestamp': timestamp
        }

        with open(filepath, 'wb') as f:
            pickle.dump(save_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

        self.logger.info(f"✓ Data saved: {filepath}")
        print(f"✓ データを保存しました: {filepath}")
        if description:
            print(f"  説明: {description}")

        return filepath

class BayesianResultsSerializer:
    """ベイズ推定結果の保存・読み込みクラス"""

    def __init__(self, save_dir: str = './bayesian_results/'):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.logger = logging.getLogger(self.__class__.__name__)

    def save_trace(self, trace: az.InferenceData,
                   linkage_type: str,
                   filename: Optional[str] = None) -> str:
        """
        ベイズ推定のトレースを保存

        Parameters
        ----------
        trace : az.InferenceData
            保存するトレース
        linkage_type : str
            'backward' または 'forward'
        filename : Optional[str]
            ファイル名（Noneの場合は自動生成）

        Returns
        -------
        str
            保存したファイルのパス
        """
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"trace_{linkage_type}_{timestamp}.nc"

        filepath = os.path.join(self.save_dir, filename)

        try:
            trace.to_netcdf(filepath)
            file_size = os.path.getsize(filepath) / (1024 * 1024)

            self.logger.info(f"✓ Trace saved: {filepath} ({file_size:.2f} MB)")
            print(f"✓ トレースを保存しました: {filepath}")
            print(f"  タイプ: {linkage_type}")
            print(f"  ファイルサイズ: {file_size:.2f} MB")

            return filepath

        except Exception as e:
            self.logger.error(f"Failed to save trace: {e}")
            raise

    def load_trace(self, filename: str) -> az.InferenceData:
        """
        保存されたトレースを読み込み

        Parameters
        ----------
        filename : str
            読み込むファイル名

        Returns
        -------
        az.InferenceData
            読み込んだトレース
        """
        if not os.path.dirname(filename):
            filepath = os.path.join(self.save_dir, filename)
        else:
            filepath = filename

        try:
            trace = az.from_netcdf(filepath)
            self.logger.info(f"✓ Trace loaded: {filepath}")
            print(f"✓ トレースを読み込みました: {filepath}")
            return trace

        except FileNotFoundError:
            self.logger.error(f"File not found: {filepath}")
            print(f"✗ ファイルが見つかりません: {filepath}")
            raise
        except Exception as e:
            self.logger.error(f"Failed to load trace: {e}")
            raise

    def save_results(self, results: BayesianModelResults,
                    linkage_type: str,
                    filename: Optional[str] = None) -> Dict[str, str]:
        """
        ベイズ推定結果を完全に保存（トレース + 統計量）

        Parameters
        ----------
        results : BayesianModelResults
            保存する推定結果
        linkage_type : str
            'backward' または 'forward'
        filename : Optional[str]
            ファイル名のベース（拡張子なし）

        Returns
        -------
        Dict[str, str]
            保存したファイルのパス辞書
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_name = filename or f"bayesian_{linkage_type}_{timestamp}"

        # トレースを保存
        trace_path = self.save_trace(
            results.trace,
            linkage_type,
            f"{base_name}_trace.nc"
        )

        # 統計情報をpickleで保存
        stats_dict = {
            'waic': results.waic,
            'loo': results.loo,
            'summary': results.summary,
            'linkage_type': linkage_type,
            'timestamp': timestamp
        }

        stats_path = os.path.join(self.save_dir, f"{base_name}_stats.pkl")
        with open(stats_path, 'wb') as f:
            pickle.dump(stats_dict, f)

        print(f"✓ 統計情報を保存しました: {stats_path}")
        waic_val = f"{results.waic:.2f}" if results.waic is not None else "N/A"
        loo_val = f"{results.loo:.2f}" if results.loo is not None else "N/A"
        print(f"  WAIC: {waic_val}")
        print(f"  LOO: {loo_val}")

        return {
            'trace': trace_path,
            'stats': stats_path
        }

print("✓ Results serialization classes defined")

In [ ]:
# ===== パイプラインに保存機能を追加 =====

class APLAnalysisPipelineWithSave(APLAnalysisPipeline):
    """保存機能付きAPL分析パイプライン"""

    def __init__(self, config: Config):
        super().__init__(config)
        self.serializer = ResultsSerializer()
        self.bayesian_serializer = BayesianResultsSerializer()

    def run_full_analysis(self, auto_save: bool = True) -> AnalysisResults:
        """
        完全な分析パイプラインを実行（自動保存オプション付き）

        Parameters
        ----------
        auto_save : bool
            分析完了後に結果を自動保存するか

        Returns
        -------
        AnalysisResults
            分析結果
        """
        # 元の分析を実行
        results = super().run_full_analysis()

        # 自動保存
        if auto_save:
            self.logger.info("\n--- Auto-saving results ---")
            self.save_results()

        return results

    def save_results(self, filename: Optional[str] = None) -> str:
        """
        現在の分析結果を保存

        Parameters
        ----------
        filename : Optional[str]
            保存ファイル名

        Returns
        -------
        str
            保存したファイルのパス
        """
        return self.serializer.save_results(self.results, filename)

    def load_results(self, filename: str) -> None:
        """
        保存済みの分析結果を読み込む

        Parameters
        ----------
        filename : str
            読み込むファイル名
        """
        self.results = self.serializer.load_results(filename)
        self.logger.info("Results loaded successfully")

    def run_bayesian_analysis(self, linkage_type: str = 'backward',
                             auto_save: bool = True) -> BayesianModelResults:
        """
        ベイズ分析を実行（自動保存オプション付き）

        Parameters
        ----------
        linkage_type : str
            'backward' または 'forward'
        auto_save : bool
            推定完了後に結果を自動保存するか

        Returns
        -------
        BayesianModelResults
            ベイズ推定結果
        """
        # 元のベイズ分析を実行
        results = super().run_bayesian_analysis(linkage_type)

        # 自動保存
        if auto_save:
            self.logger.info("\n--- Auto-saving Bayesian results ---")
            self.bayesian_serializer.save_results(results, linkage_type)

        return results

    def export_results_to_formats(self) -> Dict[str, str]:
        """
        分析結果を複数の形式でエクスポート

        Returns
        -------
        Dict[str, str]
            エクスポートしたファイルのパス辞書
        """
        self.logger.info("\n--- Exporting results to multiple formats ---")

        export_paths = {}

        # 1. Pickle形式（完全な結果）
        export_paths['pickle'] = self.save_results()

        # 2. CSV形式（地域別APL）
        back_df, forw_df = self.table_exporter.create_apl_table(
            self.results.apl_results, self.config.regions
        )
        export_paths['csv_back'] = os.path.join(
            self.config.table_dir, "regional_back_apl.csv"
        )
        export_paths['csv_forw'] = os.path.join(
            self.config.table_dir, "regional_forw_apl.csv"
        )

        # 3. Excel形式（各年の行列）
        for year in self.results.raw_linkages.keys():
            linkage = self.results.raw_linkages[year]
            io_data = self.results.io_data[year]
            self.table_exporter.save_matrices_to_excel(
                linkage.input_coefficient,
                linkage.leontief_inverse,
                io_data.labels,
                year
            )
            export_paths[f'excel_{year}'] = os.path.join(
                self.config.table_dir, f"matrix_{year}.xlsx"
            )

        print("\n=== エクスポート完了 ===")
        for key, path in export_paths.items():
            print(f"  {key}: {path}")

        return export_paths

print("✓ Pipeline with save functionality defined")

## 計算実行部分（適宜分類したい地域・産業で書き換え）

In [ ]:
# ===== APL計算実行（保存機能付き） =====

# 1. 産業分類の抽出
first_file = list(config.years.values())[0]
config.sectors = SectorExtractor.extract_from_excel(
    first_file, start_sector="農林水産業", end_sector="その他"
)

# 変数名を pipeline に統一
pipeline = APLAnalysisPipelineWithSave(config)
pipeline.loader = IOTableLoader(config.regions, config.sectors)

# 分析実行
results = pipeline.run_full_analysis(auto_save=True)

# 全形式でエクスポート
export_paths = pipeline.export_results_to_formats()

print("\n" + "=" * 60)
print("Analysis and export completed!")
print("=" * 60)

In [ ]:
# ===== ベイズ分析（キャッシュ機能付き） =====
# 設定フラグ
RUN_BAYESIAN = True        # True: 分析プロセスを有効化
FORCE_RERUN = False        # True: 保存ファイルがあっても強制的に再計算
SAVE_DIR = './bayesian_results/' # 保存先ディレクトリ
PLOT_TRACE = True          # True: トレースプロットを作成
PLOT_VARIANCE = True       # True: 分散寄与分解図を作成
PLOT_MODEL_GRAPH = True    # True: モデル構造図を作成


# 保存・読み込み用クラスのインスタンス化（セル13の定義が必要です）
bayesian_serializer = BayesianResultsSerializer(SAVE_DIR)


def get_latest_trace_file(linkage_type):
    """指定されたタイプの最新のトレースファイルを検索（ファイル名パターンを修正）"""
    # 保存名に合わせて 'bayesian_{linkage_type}_' で始まるファイルを探す
    pattern = os.path.join(SAVE_DIR, f"bayesian_{linkage_type}_*_trace.nc")
    files = sorted(glob.glob(pattern))
    return files[-1] if files else None


def get_bayesian_results(pipeline, linkage_type, force_rerun=False):
    """キャッシュを確認し、なければ計算、あれば読み込む関数"""
    latest_file = get_latest_trace_file(linkage_type)

    # 1. キャッシュが存在し、かつ強制再計算でない場合 -> 読み込み
    if latest_file and not force_rerun:
        print(f"\n=== {linkage_type}連関: 保存済みの結果が見つかりました ===")
        print(f"読み込み中: {latest_file}")

        try:
            # トレースの読み込み
            trace = bayesian_serializer.load_trace(latest_file)

            # サマリーの再計算
            summary = az.summary(trace)
            results = BayesianModelResults(
                trace=trace,
                model=None,
                waic=None,
                loo=None,
                summary=summary
            )
            print("✓ キャッシュからの読み込み完了")
            return results

        except Exception as e:
            print(f"⚠ 読み込みエラー: {e}")
            print("再計算を実行します...")

    # 2. キャッシュがない、または強制再計算の場合 -> 計算実行
    print(f"\n=== {linkage_type}連関: 新規にベイズ推定を実行します ===")

    # パイプラインで計算
    results = pipeline.run_bayesian_analysis(linkage_type)
    # 結果の保存
    bayesian_serializer.save_results(results, linkage_type)

    return results


# --- メイン実行処理 ---

if RUN_BAYESIAN:
    print("Starting Bayesian Analysis with Cache System...")

    # 後方連関の分析・取得
    bayesian_results_back = get_bayesian_results(
        pipeline, 'backward', force_rerun=FORCE_RERUN
    )
    print("\n[後方APL サマリー]")
    print(bayesian_results_back.summary)

    # トレースプロット
    if PLOT_TRACE:
        print("\n=== 後方連関MCMCトレースプロット作成中 ===")
        pipeline.visualizer.plot_mcmc_trace(
            bayesian_results_back,
            linkage_type='backward'
        )

    # 前方連関の分析・取得
    bayesian_results_forw = get_bayesian_results(
        pipeline, 'forward', force_rerun=FORCE_RERUN
    )
    print("\n[前方APL サマリー]")
    print(bayesian_results_forw.summary)

    # トレースプロット
    if PLOT_TRACE:
        print("\n=== 前方連関MCMCトレースプロット作成中 ===")
        pipeline.visualizer.plot_mcmc_trace(
            bayesian_results_forw,
            linkage_type='forward'
        )

    # ↓↓↓ 分散寄与分解図の作成 ↓↓↓
    if PLOT_VARIANCE:
        print("\n" + "="*60)
        print("=== 分散寄与分解図の作成 ===")
        print("="*60)

        # 後方APLの分散寄与
        print("\n[1/3] 後方APLの分散寄与図を作成中...")
        pipeline.visualizer.plot_variance_contribution(
            bayesian_results_back,
            linkage_type='backward',
            target_region='四国'
        )

        # 前方APLの分散寄与
        print("\n[2/3] 前方APLの分散寄与図を作成中...")
        pipeline.visualizer.plot_variance_contribution(
            bayesian_results_forw,
            linkage_type='forward',
            target_region='四国'
        )

        # 両方を並べて比較
        print("\n[3/3] 後方・前方APLの比較図を作成中...")
        pipeline.visualizer.plot_variance_comparison(
            bayesian_results_back,
            bayesian_results_forw,
            target_region='四国'
        )

        print("\n✓ 分散寄与分解図の作成が完了しました")

    # モデル構造図の作成
    if PLOT_MODEL_GRAPH:
        print("\n" + "="*60)
        print("=== ベイズモデル構造図の作成 ===")
        print("="*60)

        # 詳細版（全パラメータ表示）
        print("\n[1/2] 詳細版モデル図を作成中...")
        pipeline.visualizer.plot_model_graph('bayesian_model_detailed.png')

        # # シンプル版（論文用）
        # print("\n[2/2] シンプル版モデル図を作成中...")
        # pipeline.visualizer.plot_model_graph_simple('bayesian_model_simple.png')

        print("\n✓ モデル構造図の作成が完了しました")

    print("\n" + "="*60)
    print("Bayesian analysis completed!")
    print("="*60)

else:
    print("ベイズ分析はスキップされました")
    print("実行する場合は RUN_BAYESIAN = True に変更してください")


In [ ]:
# 1. 論文図7: 第三次産業比率との相関散布図
print("\n--- 論文図7: 相関散布図の作成 ---")
pipeline.visualizer.plot_tertiary_correlation(
    results.apl_results,
    results.tertiary_ratios,
    config,
    'backward'
)

# 2. 論文図6: 全国トレンドからの逸脱度（ベイズ分析後）
if 'bayesian_results_back' in globals():
    print("\n--- 論文図6: 逸脱度トレンドの作成 ---")
    pipeline.visualizer.plot_deviation_trend(
        bayesian_results_back,
        config,
        target_region='四国'
    )

    # 論文図4: 交互作用キャタピラー（四国）
    print("\n--- 論文図4: 四国の産業別交互作用 ---")
    pipeline.visualizer.plot_caterpillar(
        bayesian_results_back,
        '四国',
        config
    )

# 3. 論文図2: 地域を集約（9x9）したマクロ分析の再現
print("\n--- 論文図2: 地域集約マクロ分析の実行 ---")
macro_results = {}
for year, io_data in results.io_data.items():
    # 9x9行列に集約 [cite: 236]
    Z_agg, x_agg = pipeline.aggregator.aggregate_matrix_by_region(
        io_data.transaction_matrix.values,
        io_data.gross_output,
        io_data.labels,
        config.regions
    )

    # 集約行列でAPLを再計算
    # 簡略化のためDataFrameに戻して計算機へ
    Z_df = pd.DataFrame(Z_agg, index=config.regions, columns=config.regions)
    x_ser = pd.Series(x_agg, index=config.regions)
    linkage = pipeline.calculator.calculate(Z_df, x_ser)

    macro_results[year] = {
        'back': np.nanmean(linkage.apl_matrix, axis=0),
        'forw': np.nanmean(linkage.apl_matrix, axis=1)
    }

# 集約APLの軌跡をプロット
pipeline.visualizer.plot_trajectory(macro_results, config.regions, highlight_region='四国')

## GIS表示

In [ ]:
# 1. 保存ディレクトリの設定
MAP_SAVE_DIR = os.path.join(config.table_dir, 'spatial_analysis/')
os.makedirs(MAP_SAVE_DIR, exist_ok=True)

# 2. results.apl_results から地図用データを自動生成
def prepare_map_dataframe(analysis_results, regions):
    map_df = pd.DataFrame({'region': regions})
    for year in sorted(analysis_results.apl_results.keys()):
        map_df[f'後方APL_{year}'] = analysis_results.apl_results[year]['back']
        map_df[f'前方APL_{year}'] = analysis_results.apl_results[year]['forw']
    return map_df

df_apl_map = prepare_map_dataframe(results, config.regions)

# 3. 地図データの読み込みとマージ
pref_to_region = {
    '北海道': '北海道',
    '青森県': '東北', '岩手県': '東北', '宮城県': '東北', '秋田県': '東北', '山形県': '東北', '福島県': '東北',
    '茨城県': '関東', '栃木県': '関東', '群馬県': '関東', '埼玉県': '関東', '千葉県': '関東', '東京都': '関東', '神奈川県': '関東',
    '新潟県': '中部', '富山県': '中部', '石川県': '中部', '福井県': '中部', '山梨県': '中部', '長野県': '中部', '岐阜県': '中部', '静岡県': '中部', '愛知県': '中部',
    '三重県': '近畿', '滋賀県': '近畿', '京都府': '近畿', '大阪府': '近畿', '兵庫県': '近畿', '奈良県': '近畿', '和歌山県': '近畿',
    '鳥取県': '中国', '島根県': '中国', '岡山県': '中国', '広島県': '中国', '山口県': '中国',
    '徳島県': '四国', '香川県': '四国', '愛媛県': '四国', '高知県': '四国',
    '福岡県': '九州', '佐賀県': '九州', '長崎県': '九州', '熊本県': '九州', '大分県': '九州', '宮崎県': '九州', '鹿児島県': '九州',
    '沖縄県': '沖縄'
}

try:
    gdf_base = gpd.read_file(geojson_path)
    gdf_base['region'] = gdf_base['name'].map(pref_to_region)
    gdf = gdf_base.merge(df_apl_map, on='region', how='left')
    print("✓ 地図データのマージが完了しました")
except Exception as e:
    print(f"✗ マージ失敗: {e}")

# 4. 地図描画・GIF作成ロジック
def create_animated_maps(linkage_type='後方', display_inline=True):
    target_cols = [c for c in df_apl_map.columns if c.startswith(linkage_type)]
    filenames = []

    # カラー設定
    colors_list = ['#E8F4FD','#C2E3F7','#7CC7EA','#42A5F5','#1E88E5','#1565C0','#0D47A1']
    map_cmap = LinearSegmentedColormap.from_list('apl_blues', colors_list, N=256)

    # 全データ範囲からスケール固定
    all_vals = df_apl_map[target_cols].values
    vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)

    for col in target_cols:
        year_label = col.split('_')[-1]
        fig, ax = plt.subplots(figsize=(10, 10))

        # データの分離
        is_okinawa = gdf['name'].str.contains('沖縄')
        mainland = gdf[~is_okinawa]
        okinawa = gdf[is_okinawa]

        # --- 1. 本土の描画 ---
        mainland.plot(column=col, cmap=map_cmap, vmin=vmin, vmax=vmax,
                      ax=ax, edgecolor='black', linewidth=0.3)

        ax.set_xlim(128, 146)
        ax.set_ylim(30, 46)
        ax.axis('off')
        ax.set_title(f"{linkage_type}APL {year_label}", fontsize=18, fontweight='bold', pad=10)

        # --- 2. 沖縄インセット（小窓）の作成 ---
        # loc='upper left' で左上に配置。borderpadで位置を微調整
        axins = inset_axes(ax, width="25%", height="25%", loc='upper left', borderpad=2)
        axins.set_facecolor('white')

        okinawa.plot(column=col, cmap=map_cmap, vmin=vmin, vmax=vmax,
                    ax=axins, edgecolor='black', linewidth=0.5)

        # 沖縄本島周辺に拡大ズーム（座標範囲を絞る）
        axins.set_xlim(126.5, 129.0)
        axins.set_ylim(25.8, 27.5)
        axins.set_xticks([])
        axins.set_yticks([])

        # 境界線の設定：左と上を消し、右と下（本土側）のみを表示して仕切り線にする
        for side in ['left', 'top']:
            axins.spines[side].set_visible(False)
        for side in ['right', 'bottom']:
            axins.spines[side].set_visible(True)
            axins.spines[side].set_edgecolor('black')
            axins.spines[side].set_linewidth(1.5)

        # --- 3. カラーバー ---
        sm = plt.cm.ScalarMappable(cmap=map_cmap, norm=Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
        cbar.set_label(f'{linkage_type}APL', fontsize=12)

        # 保存
        fname = os.path.join(MAP_SAVE_DIR, f"{linkage_type}_map_{year_label}.png")
        plt.savefig(fname, bbox_inches='tight', dpi=150, facecolor='white')
        filenames.append(fname)
        plt.close()

    # GIF作成
    gif_name = os.path.join(MAP_SAVE_DIR, f'complete_{linkage_type}_apl_animation.gif')
    with imageio.get_writer(gif_name, mode='I', duration=1500, loop=0) as writer:
        for f in filenames:
            writer.append_data(imageio.imread(f))

    print(f"✓ {linkage_type}APLアニメーション作成完了: {gif_name}")
    if display_inline:
        display(IPImage(filename=gif_name))
    return gif_name

# 実行
gif_back = create_animated_maps('後方')
gif_forw = create_animated_maps('前方')